# Ablation

In [ ]:
import os
import sys
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from sklearn.metrics import accuracy_score, roc_auc_score
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv, global_mean_pool, TopKPooling
from torch_geometric.loader import DataLoader
from torch_geometric.explain import Explainer, PGExplainer
from torch_geometric.utils import subgraph
from tqdm import tqdm

sys.path.append(os.path.abspath('..'))
from src.explainability import compute_final_fidelity_scores

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')

CONFIGS = {
    'M1': {'FLR': True,  'GLR': False, 'ELR': False},
    'M2': {'FLR': False, 'GLR': True,  'ELR': False},
    'M3': {'FLR': False, 'GLR': False, 'ELR': True},
    'M4': {'FLR': True,  'GLR': True,  'ELR': False},
    'M5': {'FLR': True,  'GLR': False, 'ELR': True},
    'M6': {'FLR': False, 'GLR': True,  'ELR': True},
    'M7': {'FLR': True,  'GLR': True,  'ELR': True},
    'M8': {'FLR': False, 'GLR': False, 'ELR': False} 
}

class AblationGraphSAGE(nn.Module):
    def __init__(self, config):
        super().__init__()
        in_c = 128 if config['FLR'] else 576 
        self.conv1 = SAGEConv(in_c, 64)
        self.conv2 = SAGEConv(64, 64)
        
        self.use_elr = config['ELR']
        if self.use_elr:
            self.pool = TopKPooling(64, ratio=0.5)
            
        self.lin1 = nn.Linear(64, 32)
        self.lin2 = nn.Linear(32, 2)

    def forward(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        if self.use_elr:
            x, edge_index, _, batch, _, _ = self.pool(x, edge_index, None, batch)
        x = global_mean_pool(x, batch)
        x = F.relu(self.lin1(x))
        return self.lin2(x)

def evaluate_model_fidelity(model, test_loader):
    model.eval()
    
    explainer = Explainer(
        model=model,
        algorithm=PGExplainer(epochs=30, lr=0.003),
        explanation_type='phenomenon',
        edge_mask_type='object',
        model_config=dict(
            mode='multiclass_classification',
            task_level='graph',
            return_type='raw',
        ),
    )

    explainer.algorithm.to(device)

    sparsity_levels = np.linspace(0.05, 0.95, 19)
    raw_p_delete = {round(s, 2): [] for s in sparsity_levels}
    raw_p_retain = {round(s, 2): [] for s in sparsity_levels}
    
    print("Evaluating Fidelity on Test Set...")
    for data in test_loader:
        data = data.to(device)
        
        with torch.no_grad():
            orig_out = model(data.x, data.edge_index, data.batch)
            target_class = orig_out.argmax(dim=1)[0]
            
        explanation = explainer(data.x, data.edge_index, target=target_class.unsqueeze(0), batch=data.batch)
        edge_mask = explanation.edge_mask
        
        node_scores = torch.zeros(data.num_nodes, device=device)
        if data.edge_index.numel() > 0:
            for i, edge in enumerate(data.edge_index.T):
                node_scores[edge[0]] = torch.max(node_scores[edge[0]], edge_mask[i])
                node_scores[edge[1]] = torch.max(node_scores[edge[1]], edge_mask[i])
            
        for s in sparsity_levels:
            s_rounded = round(s, 2)
            num_nodes_to_keep = max(1, int((1 - s) * data.num_nodes))
            
            _, ranked_indices = torch.sort(node_scores, descending=True)
            
            keep_indices_plus = ranked_indices[int(s * data.num_nodes):]
            if len(keep_indices_plus) == 0: 
                keep_indices_plus = ranked_indices[-1:] 
                
            keep_indices_minus = ranked_indices[:num_nodes_to_keep]
            
            edge_index_plus, _ = subgraph(keep_indices_plus, data.edge_index, relabel_nodes=True)
            x_plus = data.x[keep_indices_plus]
            batch_plus = torch.zeros(x_plus.size(0), dtype=torch.long, device=device)
            with torch.no_grad():
                out_plus = model(x_plus, edge_index_plus, batch_plus)
                p_del = F.softmax(out_plus, dim=1)[0, target_class].item()
                raw_p_delete[s_rounded].append(p_del)

            edge_index_minus, _ = subgraph(keep_indices_minus, data.edge_index, relabel_nodes=True)
            x_minus = data.x[keep_indices_minus]
            batch_minus = torch.zeros(x_minus.size(0), dtype=torch.long, device=device)
            with torch.no_grad():
                out_minus = model(x_minus, edge_index_minus, batch_minus)
                p_ret = F.softmax(out_minus, dim=1)[0, target_class].item()
                raw_p_retain[s_rounded].append(p_ret)

    avg_p_delete = {s: np.mean(vals) for s, vals in raw_p_delete.items()}
    avg_p_retain = {s: np.mean(vals) for s, vals in raw_p_retain.items()}
    
    return avg_p_delete, avg_p_retain

if __name__ == "__main__":
    models_to_test = ["M1", "M2", "M3", "M4", "M5", "M6", "M7", "M8"]
    os.makedirs('../results', exist_ok=True) 
    
    # test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)
    
    for model_id in models_to_test:
        print(f"\n======================================")
        print(f"--- Evaluating {model_id} ---")
        config = CONFIGS[model_id]
        
        model = AblationGraphSAGE(config)
        try:
            model.load_state_dict(torch.load(f"../models/{model_id}.pth", map_location=device))
            model.to(device)
        except FileNotFoundError:
            print(f"  [X] Could not find ../models/{model_id}.pth, skipping...")
            continue
            
        print("Evaluating Predictive Correctness...")
        model.eval()
        all_true = []
        all_probs = []
        all_preds = []
        
        with torch.no_grad():
            for data in test_loader: 
                data = data.to(device)
                out = model(data.x, data.edge_index, data.batch)
                probs = F.softmax(out, dim=1)[:, 1] # Probability of Malignant (1)
                preds = out.argmax(dim=1)
                
                all_true.extend(data.y.cpu().tolist())
                all_probs.extend(probs.cpu().tolist())
                all_preds.extend(preds.cpu().tolist())
                
        acc = accuracy_score(all_true, all_preds)
        auc = roc_auc_score(all_true, all_probs)
        print(f"Accuracy: {acc:.4f} | AUC: {auc:.4f}")
        
        results_data = {
            'model_id': model_id,
            'true': all_true,
            'probs': all_probs,
            'auc': auc
        }
        with open(f'../results/{model_id}_results.json', 'w') as f:
            json.dump(results_data, f)
            
        p_del, p_ret = evaluate_model_fidelity(model, test_loader)
        
        print(f"\n{model_id} FINAL ARRAYS:")
        print(f"Fid+ (p_delete) array: {list(p_del.values())}")
        print(f"Fid- (p_retain) array: {list(p_ret.values())}")